# SVR Phase 2 | MARBERTv2 single-task training

I fine-tune one classifier for one voicemail field and seed at a time. I use the frozen train and validation tables and save checkpoints and results to Google Drive.

1. Mount Drive and set paths.
2. Install the team dependencies in an isolated Python 3.11 environment.
3. Validate the snapshot and run a GPU smoke test.
4. Opt in to one training invocation.
5. Assemble finished seeds and compare with TF-IDF.

I upload the reviewed code and frozen inputs to private Drive folders before running setup. See `reports/marbertv2_phase2_report.md` for the exact folder layout, resume procedure, and optional learning-rate sweep.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
CODE_ROOT = DRIVE / "SVR/phase2-code"
TRAIN_CSV = DRIVE / "SVR/frozen-v2/clean_training_train.csv"
VALIDATION_CSV = DRIVE / "SVR/frozen-v2/clean_training_val.csv"
CLASS_WEIGHTS = DRIVE / "SVR/frozen-v2/class_weights.json"
OUTPUT_ROOT = DRIVE / "SVR/marbertv2-output"
SHARED_CONFIG = CODE_ROOT / "notebooks/training/shared_config.json"
ENTRY_POINT = CODE_ROOT / "notebooks/training/marbertv2_phase2.py"

for path in (TRAIN_CSV, VALIDATION_CSV, CLASS_WEIGHTS, SHARED_CONFIG, ENTRY_POINT,
             CODE_ROOT / "infra/harness.py", CODE_ROOT / "requirements/base.txt",
             CODE_ROOT / "requirements/lama.txt"):
    if not path.is_file():
        raise FileNotFoundError(path)

In [ ]:
# Isolate the pinned team stack from the Colab kernel packages.
gpu_smoke_ok = False
work = Path("/content/svr-marbertv2")
work.mkdir(exist_ok=True)
env = dict(os.environ, UV_CACHE_DIR=str(work / "uv-cache"),
           UV_PYTHON_INSTALL_DIR=str(work / "python"), HF_HOME=str(work / "hf-cache"),
           HF_HUB_DISABLE_TELEMETRY="1", TOKENIZERS_PARALLELISM="false")
bootstrap = work / "bootstrap"
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--target", str(bootstrap), "uv"], env=env, check=True)
uv = bootstrap / "bin/uv"
venv = work / "venv"
subprocess.run([str(uv), "venv", "--allow-existing", "--managed-python", "--python", "3.11", str(venv)],
               env=env, check=True)
python = venv / "bin/python"
subprocess.run([str(uv), "pip", "install", "--python", str(python),
                "-r", str(CODE_ROOT / "requirements/base.txt"),
                "-r", str(CODE_ROOT / "requirements/lama.txt")], env=env, check=True)
env["PYTHONPATH"] = str(CODE_ROOT)

def phase2(action, **options):
    args = [str(python), "-u", str(ENTRY_POINT), action,
            "--shared", str(SHARED_CONFIG), "--train", str(TRAIN_CSV),
            "--validation", str(VALIDATION_CSV), "--weights", str(CLASS_WEIGHTS),
            "--output", str(OUTPUT_ROOT)]
    for name, value in options.items():
        flag = "--" + name.replace("_", "-")
        if value is True:
            args.append(flag)
        elif value is not None and value is not False:
            args.extend([flag, str(value)])
    subprocess.run(args, cwd=work, env=env, check=True)

In [ ]:
# If either command fails, I stop and read the original traceback above the subprocess error.
gpu_smoke_ok = False
phase2("check")
phase2("smoke", field="complaint_category", seed=42)
gpu_smoke_ok = True

## Train one field and seed

I keep `RUN_ID` fixed across the 20 task/seed pairs for one configuration. I iterate through seeds in order **42, 1337, 2023, 7, 99**, with one invocation per field per seed. Supported fields are `complaint_category`, `complaint_urgency`, `customer_state`, and `churn_risk`. I set `RESUME = True` to continue an interrupted pair from its last completed epoch on the same GPU type and package versions.

In [ ]:
RUN_ID = "marbertv2-lr2e5"
FIELD = "complaint_category"
SEED = 42
LEARNING_RATE = 2e-5
EPOCHS = 3
REFERENCE_RUN_ID = None
RESUME = False
RUN_TRAINING = False

if RUN_TRAINING:
    if not gpu_smoke_ok:
        raise RuntimeError("Complete the frozen-input and GPU smoke checks in this runtime first.")
    phase2("train", run_id=RUN_ID, field=FIELD, seed=SEED,
           learning_rate=LEARNING_RATE, epochs=EPOCHS, resume=RESUME)
else:
    print("Training is off. Set RUN_TRAINING = True for this field and seed.")

## Assemble results

I can assemble results after each seed. Once all four tasks for a seed finish, this step writes the official four-field prediction file and validation metrics. The run record and Markdown comparison are generated after all five seeds finish. I keep the same run ID and settings used for training.

In [ ]:
WRITE_RESULTS = False

if WRITE_RESULTS:
    phase2("finalize", run_id=RUN_ID, learning_rate=LEARNING_RATE, epochs=EPOCHS,
           reference_run_id=REFERENCE_RUN_ID)
else:
    print("Artifact assembly is off. Set WRITE_RESULTS = True to assemble completed seeds.")